# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [2]:
import sys
print(sys.executable)

/Users/yexuanhe/ai/projects/tinyml-arduino/bin/python


In [3]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [6]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [7]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.
X = wine.data
y = wine.target

In [8]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [9]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.
num_classes = 3

y_train = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

In [11]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [12]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/20
13/13 [==============================] - 0s 6ms/step - loss: 1.0389 - accuracy: 0.4242 - val_loss: 0.9025 - val_accuracy: 0.6000
Epoch 2/20
13/13 [==============================] - 0s 1ms/step - loss: 0.7625 - accuracy: 0.7475 - val_loss: 0.6922 - val_accuracy: 0.8400
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 0.5642 - accuracy: 0.8687 - val_loss: 0.5202 - val_accuracy: 0.8800
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 0.3999 - accuracy: 0.9495 - val_loss: 0.3880 - val_accuracy: 0.8800
Epoch 5/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2798 - accuracy: 0.9798 - val_loss: 0.2900 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1976 - accuracy: 0.9899 - val_loss: 0.2226 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1481 - accuracy: 0.9899 - val_loss: 0.1752 - val_accuracy: 0.9600
Epoch 8/20
13/13 [==

In [13]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Test Accuracy: 1.0000
2/2 [==============================] - 0s 962us/step

Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        19
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [14]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = "model_base.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"TFLite model saved as '{tflite_path}'")
print(f"Model size:", round(len(tflite_model) / 1024, 2), "KB")

INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpjkc1gwtm/assets


INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpjkc1gwtm/assets
2026-05-20 21:23:18.164256: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:23:18.164280: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:23:18.164516: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpjkc1gwtm
2026-05-20 21:23:18.164975: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:23:18.164981: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpjkc1gwtm
2026-05-20 21:23:18.166206: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled


TFLite model saved as 'model_base.tflite'
Model size: 14.07 KB


2026-05-20 21:23:18.169806: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:23:18.203231: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpjkc1gwtm
2026-05-20 21:23:18.209174: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 44660 microseconds.
2026-05-20 21:23:18.216915: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [43]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale      = input_details['quantization'][0]
    input_zero_point = input_details['quantization'][1]
    output_scale      = output_details['quantization'][0]
    output_zero_point = output_details['quantization'][1]

    y_true = np.argmax(y_test_cat, axis=1)
    y_pred = []

    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)

        # Quantize input if needed
        if input_details['dtype'] == np.int8:
            sample = (sample / input_scale + input_zero_point).astype(np.int8)

        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])

        # Dequantize output if needed
        if output_details['dtype'] == np.int8:
            output = (output.astype(np.float32) - output_zero_point) * output_scale

        y_pred.append(np.argmax(output))

    y_pred = np.array(y_pred)

    # Step 4: Report results.
    print(f"Model size:", round(len(tflite_model) / 1024, 2), "KB")
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=wine.target_names))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [44]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'
quantize_and_evaluate(model, X_test, y_test, quant_type='int8',     filename='model_int8.tflite')
quantize_and_evaluate(model, X_test, y_test, quant_type='float16',  filename='model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test, quant_type='dynamic',  filename='model_dynamic.tflite')


INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpv2cspepy/assets


INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpv2cspepy/assets


Model size: 5.74 KB

Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        19
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


/Users/yexuanhe/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 21:39:33.491790: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:39:33.491811: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:39:33.491967: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpv2cspepy
2026-05-20 21:39:33.492389: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:39:33.492394: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpv2cspepy
2026-05-20 21:39:33.493703: I tensorflow/cc/saved_model/loader.cc:233]

INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpdy780_m4/assets


...

2026-05-20 21:39:33.760086: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:39:33.760100: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:39:33.760183: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpdy780_m4
2026-05-20 21:39:33.760595: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:39:33.760600: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpdy780_m4
2026-05-20 21:39:33.761705: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:39:33.781128: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpdy780_m4
2026-05-

INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp2f_jaif1/assets


INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp2f_jaif1/assets
2026-05-20 21:39:34.091423: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:39:34.091435: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.


Model size: 8.17 KB

Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        19
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-05-20 21:39:34.091539: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp2f_jaif1
2026-05-20 21:39:34.091951: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:39:34.091956: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp2f_jaif1
2026-05-20 21:39:34.093071: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:39:34.112200: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp2f_jaif1
2026-05-20 21:39:34.117446: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 25907 microseconds.


## Problem 1 - Part (c)

### Pruning

In [17]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

initial_sparsity = 0.5
final_sparsity   = 0.7
batch_size       = 8
epochs           = 20
dataset_size     = len(X_train)

end_step = int(dataset_size / batch_size * epochs)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=initial_sparsity,
    final_sparsity=final_sparsity,
    begin_step=0,
    end_step=end_step
)

In [18]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruning_params = {'pruning_schedule': pruning_schedule}

pruned_model = tf.keras.Sequential([
    prune_low_magnitude(tf.keras.layers.Dense(64, activation='relu',
                        input_shape=(X_train.shape[1],)), **pruning_params),
    prune_low_magnitude(tf.keras.layers.Dense(32, activation='relu'), **pruning_params),
    prune_low_magnitude(tf.keras.layers.Dense(num_classes, activation='softmax'), **pruning_params)
])

In [19]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list
pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

pruning_history = pruned_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()]
)

Epoch 1/10
13/13 [==============================] - 1s 6ms/step - loss: 1.0478 - accuracy: 0.4747 - val_loss: 0.8821 - val_accuracy: 0.6000
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7779 - accuracy: 0.7576 - val_loss: 0.6800 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5763 - accuracy: 0.9495 - val_loss: 0.5131 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4181 - accuracy: 0.9899 - val_loss: 0.3900 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.3028 - accuracy: 0.9899 - val_loss: 0.2969 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2218 - accuracy: 0.9798 - val_loss: 0.2246 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1646 - accuracy: 0.9798 - val_loss: 0.1788 - val_accuracy: 0.9600
Epoch 8/10
13/13 [==

In [20]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned_model)

print("Model size:", round(len(tflite_pruned_model) / 1024, 2), "KB")

INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpnjmbxxfw/assets


INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpnjmbxxfw/assets


Model size: 14.14 KB


2026-05-20 21:23:20.334318: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:23:20.334333: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:23:20.334429: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpnjmbxxfw
2026-05-20 21:23:20.334702: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:23:20.334706: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpnjmbxxfw
2026-05-20 21:23:20.335354: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:23:20.342303: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpnjmbxxfw
2026-05-

In [21]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_pred = np.argmax(stripped_model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 1ms/step
Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        19
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [22]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)
student_model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [23]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels
teacher_preds_soft = model.predict(X_train)

4/4 [==============================] - 0s 649us/step


In [24]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

y_train_combined = np.concatenate([y_train, teacher_preds_soft], axis=1)
alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :3]   # One-hot ground truth labels
    y_true_soft = y_true_combined[:, 3:]   # Teacher soft predictions

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1 - alpha) * soft_loss

In [25]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2
student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

kd_history = student_model.fit(
    X_train, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/10
13/13 [==============================] - 0s 5ms/step - loss: 1.2097 - accuracy: 0.3939 - val_loss: 1.1281 - val_accuracy: 0.6000
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 1.0196 - accuracy: 0.6061 - val_loss: 0.9697 - val_accuracy: 0.6800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.8844 - accuracy: 0.6869 - val_loss: 0.8565 - val_accuracy: 0.6800
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7793 - accuracy: 0.7172 - val_loss: 0.7554 - val_accuracy: 0.7200
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6901 - accuracy: 0.7374 - val_loss: 0.6827 - val_accuracy: 0.7600
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6184 - accuracy: 0.8081 - val_loss: 0.6087 - val_accuracy: 0.8400
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5477 - accuracy: 0.8485 - val_loss: 0.5445 - val_accuracy: 0.8400
Epoch 8/10
13/13 [==

In [26]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd_model)

print("Model size:", round(len(tflite_kd_model) / 1024, 2), "KB")

INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp_fapt4p9/assets


INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp_fapt4p9/assets


Model size: 6.1 KB


2026-05-20 21:23:21.017109: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:23:21.017122: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:23:21.017232: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp_fapt4p9
2026-05-20 21:23:21.017669: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:23:21.017674: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp_fapt4p9
2026-05-20 21:23:21.018842: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:23:21.037999: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmp_fapt4p9
2026-05-

In [27]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix
y_pred = np.argmax(student_model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 1ms/step

Classification Report:
              precision    recall  f1-score   support

     class_0       0.90      1.00      0.95        19
     class_1       1.00      1.00      1.00        21
     class_2       1.00      0.86      0.92        14

    accuracy                           0.96        54
   macro avg       0.97      0.95      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 2  0 12]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best? Pruned had the smallest size; they all performed well, just KD with reltively lower 0.96.

2. **Propose a strategy** that combines or enhances techniques learned so far. Combining KD + Int8

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why. With the combo, the size was reduced to 3.62 kB with 0.96 accuracy.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference. KD builds the model with fewer neurons and fewer weight, combining with int8 that compress the weights remain by 4 times. 


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [47]:
# Re-use the trained student_model from part (d)
# Apply int8 quantization to the already-compact student architecture (32→16→3)

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)

# Int8 quantization settings
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_kd_int8 = converter.convert()

with open("model_kd_int8.tflite", "wb") as f:
    f.write(tflite_kd_int8)

print("KD + Int8 model size:", round(len(tflite_kd_int8) / 1024, 2), "KB")

# TFLite inference with int8 quantization
interpreter = tf.lite.Interpreter(model_path="model_kd_int8.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale       = input_details['quantization'][0]
input_zero_point  = input_details['quantization'][1]
output_scale      = output_details['quantization'][0]
output_zero_point = output_details['quantization'][1]

y_true = np.argmax(y_test, axis=1)
y_pred = []

for i in range(len(X_test)):
    sample = X_test[i:i + 1].astype(np.float32)

    # Quantize input
    sample = (sample / input_scale + input_zero_point).astype(np.int8)

    interpreter.set_tensor(input_details['index'], sample)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details['index'])

    # Dequantize output
    output = (output.astype(np.float32) - output_zero_point) * output_scale
    y_pred.append(np.argmax(output))

y_pred = np.array(y_pred)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpowcdr7g8/assets


INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpowcdr7g8/assets
/Users/yexuanhe/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


KD + Int8 model size: 3.62 KB

Classification Report:
              precision    recall  f1-score   support

     class_0       0.90      1.00      0.95        19
     class_1       1.00      1.00      1.00        21
     class_2       1.00      0.86      0.92        14

    accuracy                           0.96        54
   macro avg       0.97      0.95      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 2  0 12]]


2026-05-20 21:47:00.172552: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:47:00.172598: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:47:00.174899: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpowcdr7g8
2026-05-20 21:47:00.175363: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:47:00.175368: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpowcdr7g8
2026-05-20 21:47:00.183159: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:47:00.238206: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpowcdr7g8
2026-05-

In [48]:
# Strip pruning wrappers first (already done, but just in case)
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Apply int8 quantization to the stripped pruned model
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_pruned_int8 = converter.convert()

with open("model_pruned_int8.tflite", "wb") as f:
    f.write(tflite_pruned_int8)

print("Pruned + Int8 model size:", round(len(tflite_pruned_int8) / 1024, 2), "KB")

# Evaluate
interpreter = tf.lite.Interpreter(model_path="model_pruned_int8.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale      = input_details['quantization'][0]
input_zero_point = input_details['quantization'][1]
output_scale      = output_details['quantization'][0]
output_zero_point = output_details['quantization'][1]

y_true = np.argmax(y_test, axis=1)
y_pred = []

for i in range(len(X_test)):
    sample = X_test[i:i + 1].astype(np.float32)

    if input_details['dtype'] == np.int8:
        sample = (sample / input_scale + input_zero_point).astype(np.int8)

    interpreter.set_tensor(input_details['index'], sample)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details['index'])

    if output_details['dtype'] == np.int8:
        output = (output.astype(np.float32) - output_zero_point) * output_scale

    y_pred.append(np.argmax(output))

y_pred = np.array(y_pred)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpe9ntbjys/assets


INFO:tensorflow:Assets written to: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpe9ntbjys/assets
/Users/yexuanhe/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 21:58:41.485150: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.


Pruned + Int8 model size: 5.8 KB

Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        19
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-05-20 21:58:41.485341: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:58:41.486661: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpe9ntbjys
2026-05-20 21:58:41.487008: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:58:41.487012: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpe9ntbjys
2026-05-20 21:58:41.493417: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:58:41.532851: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/63/jrf6jmbs55q8h5qybk1k2bwc0000gn/T/tmpe9ntbjys
2026-05-20 21:58:41.537062: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
